In [ ]:
import sys, os
from pathlib import Path

# src_path parent must be in sys.path so 'from src.io import ...' resolves
src_path = Path("../../src").resolve()
sys.path.insert(0, str(src_path.parent))

import yaml
cfg_path = src_path.parent / "configs/global.yml"
cfg = yaml.safe_load(open(cfg_path))

SEED      = cfg["seed"]
PROCESSED = (src_path.parent / cfg["paths"]["processed"]).resolve()
RAW       = (src_path.parent / cfg["paths"]["raw"]).resolve()

RAW_FRAMES_DIR = RAW / "hotspot_frames"
ANN_FRAMES_DIR = PROCESSED / "hotspot_frames"

RAW_FRAMES_DIR.mkdir(parents=True, exist_ok=True)
ANN_FRAMES_DIR.mkdir(parents=True, exist_ok=True)

os.environ['MAPILLARY_ACCESS_TOKEN'] = 'MLY|27306421035677736|d506902d0cd89d1953174e7427290ac0'

print(f"PROCESSED : {PROCESSED}")
print(f"RAW       : {RAW}")
print(f"Seed      : {SEED}")

In [ ]:
# ============================================================
# API KEYS — all tokens in one place
# ============================================================

# Mapillary (already set above via os.environ, kept here for visibility)
MAPILLARY_TOKEN = os.environ.get('MAPILLARY_ACCESS_TOKEN', 'MLY|27306421035677736|d506902d0cd89d1953174e7427290ac0')

# Plate Recognizer (ParkPow's LPR API) — sign up free at https://platerecognizer.com
# Free trial: 2,500 calls/month, no credit card required
PLATE_RECOGNIZER_TOKEN = "1146fa88d4af1b8bee41c4e9a465f2db2a4c3fde"  # <-- replace this

PLATE_RECOGNIZER_API = "https://api.platerecognizer.com/v1/plate-reader/"

# Region hint for Indian plates — tells the API to prioritise IN plate formats
# (improves accuracy on Karnataka plates like KA-01-AB-1234)
PLATE_REGION = "in"  # ISO 3166-1 alpha-2 for India

print(f"Mapillary token  : {'SET' if MAPILLARY_TOKEN else 'MISSING'}")
print(f"PlateRecognizer  : {'SET (real)' if PLATE_RECOGNIZER_TOKEN != 'YOUR_PLATE_RECOGNIZER_TOKEN' else 'NOT SET — mock mode will be used'}")


In [ ]:
import importlib, subprocess, sys

if importlib.util.find_spec("ultralytics") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ultralytics", "-q"])

print("ultralytics ready")


In [ ]:
import json, requests
import matplotlib.pyplot as plt
from PIL import Image
from io import BytesIO
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

MAPILLARY_TOKEN = "MLY|27306421035677736|d506902d0cd89d1953174e7427290ac0"
print("All imports OK | Token loaded")

In [ ]:
# YOLOv8s — public model, auto-downloads (~22MB)
# Classes relevant to us: car, motorcycle, bus, truck, bicycle
model = YOLO("yolov8s.pt")
print(f"Model loaded: yolov8s")
print(f"Classes ({len(model.names)}): {list(model.names.values())}")


In [ ]:
with open(PROCESSED / "hotspots_summary.json") as f:
    hotspots = json.load(f)

top10 = sorted(hotspots, key=lambda x: x["congestion_impact_score"], reverse=True)[:10]

print("Top 10 clusters targeted:")
for h in top10:
    print(f"  [{h['cluster_id']}] {h['cluster_name']}  "
          f"score={h['congestion_impact_score']}  "
          f"lat={h['center_latitude']:.5f}  lon={h['center_longitude']:.5f}")


In [ ]:
MAPILLARY_API = "https://graph.mapillary.com/images"

def fetch_mapillary_frame(lat, lon, cluster_id, token, delta=0.001):
    for d in [delta, delta * 3]:
        bbox = f"{lon-d},{lat-d},{lon+d},{lat+d}"
        params = {
            "access_token": token,
            "fields": "id,thumb_2048_url,captured_at",
            "bbox": bbox,
            "is_pano": "false",
            "limit": 5,
        }
        resp = requests.get(MAPILLARY_API, params=params, timeout=15)
        resp.raise_for_status()
        data = resp.json().get("data", [])
        if data:
            data.sort(key=lambda x: x.get("captured_at", 0), reverse=True)
            img_url = data[0]["thumb_2048_url"]
            img_resp = requests.get(img_url, timeout=30)
            img_resp.raise_for_status()
            img = Image.open(BytesIO(img_resp.content)).convert("RGB")
            save_path = RAW_FRAMES_DIR / f"{cluster_id}_raw.jpg"
            img.save(save_path)
            return str(save_path), data[0]["id"], d
    return None, None, None


frame_meta = {}
for h in top10:
    cid = h["cluster_id"]
    print(f"Fetching [{cid}] {h['cluster_name']} ...", end=" ", flush=True)
    path, img_id, bbox_d = fetch_mapillary_frame(
        h["center_latitude"], h["center_longitude"], cid, MAPILLARY_TOKEN
    )
    if path:
        frame_meta[cid] = {"raw_path": path, "mapillary_image_id": img_id, "bbox_delta": bbox_d}
        print(f"OK  (bbox ±{bbox_d}°)")
    else:
        frame_meta[cid] = {"raw_path": None, "mapillary_image_id": None}
        print("NO IMAGE — skipped")

found = sum(1 for v in frame_meta.values() if v["raw_path"])
print(f"\nFrames fetched: {found}/{len(top10)}")


In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

CONF = 0.40
DEVICE = 0 if torch.cuda.is_available() else "cpu"  # 0 = first GPU (RTX 4050)
detection_results = {}

for cid, meta in frame_meta.items():
    if not meta["raw_path"]:
        detection_results[cid] = None
        continue

    r = model.predict(
        source=meta["raw_path"],
        conf=CONF,
        device=DEVICE,
        save=False,
        verbose=False
    )[0]

    ann_path = ANN_FRAMES_DIR / f"{cid}_annotated.jpg"
    Image.fromarray(r.plot()[..., ::-1]).save(ann_path)

    class_counts = {}
    for cls_idx in r.boxes.cls.int().tolist():
        label = model.names[cls_idx]
        class_counts[label] = class_counts.get(label, 0) + 1

    confs  = r.boxes.conf.tolist()
    total  = len(confs)
    avg_cf = round(sum(confs) / total, 3) if total else 0.0

    detection_results[cid] = {
        "annotated_image_path": f"hotspot_frames/{cid}_annotated.jpg",
        "frame_source": "Mapillary",
        "mapillary_image_id": meta["mapillary_image_id"],
        "total_vehicles_detected": total,
        "vehicle_breakdown": class_counts,
        "detection_confidence_avg": avg_cf,
    }
    print(f"[{cid}] {total} vehicles | {class_counts} | conf {avg_cf}")

print("\nInference complete.")


In [ ]:
# ============================================================
# PLATE RECOGNIZER (ParkPow LPR API)
# Reads license plates from each hotspot's annotated frame.
# Falls back to a mock result if token is not set, so the
# rest of the pipeline keeps running during local testing.
# API docs: https://guides.platerecognizer.com/docs/snapshot/api-reference
# ============================================================

def read_plates_from_image(image_path: str, token: str, region: str = "in"):
    """
    POST image to Plate Recognizer Snapshot API.
    Returns list of dicts: [{plate, score, dscore, box}, ...]
    score  = character-level reading confidence (0-1)
    dscore = plate detection confidence (0-1)
    """
    if token == "1146fa88d4af1b8bee41c4e9a465f2db2a4c3fde":
        # Mock mode — returns a plausible Karnataka plate so downstream code works
        print(f"  [MOCK] No token set — returning mock plate for {image_path}")
        return [{
            "plate": "KA01AB1234",
            "score": 0.0,          # 0 score signals this is a mock, not a real read
            "dscore": 0.0,
            "box": {},
            "mock": True
        }]

    try:
        with open(image_path, "rb") as fp:
            response = requests.post(
                PLATE_RECOGNIZER_API,
                files={"upload": fp},
                data={"regions": region},
                headers={"Authorization": f"Token {token}"},
                timeout=20,
            )
        response.raise_for_status()
        results = response.json().get("results", [])
        return [
            {
                "plate":  r["plate"].upper(),
                "score":  round(r["score"], 3),    # character-level confidence
                "dscore": round(r["dscore"], 3),   # detection confidence
                "box":    r["box"],
                "mock":   False
            }
            for r in results
        ]
    except requests.exceptions.RequestException as e:
        print(f"  [ERROR] Plate Recognizer API call failed: {e}")
        return []


# Cross-reference against violation CSV: is this plate a known offender?
import pandas as pd

_vdf = pd.read_csv(RAW / "raw.csv", usecols=["vehicle_number"], low_memory=False)
_vdf["vehicle_number"] = _vdf["vehicle_number"].str.upper().str.replace(r"[^A-Z0-9]", "", regex=True)
KNOWN_OFFENDERS = _vdf["vehicle_number"].value_counts().to_dict()
print(f"Loaded {len(KNOWN_OFFENDERS):,} unique plates from violation CSV for cross-reference")


def offender_flag(plate: str) -> dict:
    """Returns repeat-offender metadata for a plate, or zeros if not found."""
    clean = plate.upper().replace(" ", "").replace("-", "")
    count = KNOWN_OFFENDERS.get(clean, 0)
    return {
        "prior_violations": count,
        "is_repeat_offender": count > 1,
        "offender_tier": "HIGH" if count >= 5 else "MEDIUM" if count >= 2 else "NONE",
    }


# Run plate recognition on every frame that YOLO successfully processed
plate_results = {}

for cid, det in detection_results.items():
    if not det or not det.get("annotated_image_path"):
        plate_results[cid] = []
        continue

    img_path = ANN_FRAMES_DIR / f"{cid}_annotated.jpg"
    print(f"[{cid}] Reading plates ...", end=" ")

    plates = read_plates_from_image(str(img_path), PLATE_RECOGNIZER_TOKEN, PLATE_REGION)

    # Attach offender cross-reference to each plate
    for p in plates:
        p.update(offender_flag(p["plate"]))

    plate_results[cid] = plates
    plate_strs = [f"{p['plate']} (conf={p['score']}, prior={p['prior_violations']})" for p in plates]
    print(f"{len(plates)} plate(s): {plate_strs if plates else 'none'}")

print("\nPlate recognition complete.")


In [ ]:
# ============================================================
# VIOLATION CONFIDENCE SCORE
# Combines four real signals into a single 0-100 score.
# Formula and weights are explicit so they can be defended.
# ============================================================

VEHICLE_FOOTPRINT_WEIGHT = {
    "car": 2.5, "motorcycle": 1.0, "bicycle": 1.0,
    "bus": 5.0, "truck": 5.5, "van": 2.5,
}

def compute_confidence_score(cluster_id, det: dict, plates: list) -> dict:
    """
    Signal 1 — YOLO vehicle detection confidence       (weight: 35%)
      avg detection confidence across all detected vehicles

    Signal 2 — Vehicle footprint / blockage severity   (weight: 20%)
      larger vehicles cause more carriageway obstruction

    Signal 3 — Plate read confidence                   (weight: 20%)
      Plate Recognizer character-level score; 0 if no plate read

    Signal 4 — Hotspot EPI (congestion_impact_score)   (weight: 25%)
      a submission from a known CRITICAL hotspot is inherently
      more likely to reflect a real congestion-causing violation
    """
    # Signal 1: avg YOLO detection confidence
    yolo_conf = det.get("detection_confidence_avg", 0.0)  # already 0-1

    # Signal 2: vehicle footprint (normalised to 0-1 against max footprint of 5.5)
    breakdown = det.get("vehicle_breakdown", {})
    if breakdown:
        weights = [VEHICLE_FOOTPRINT_WEIGHT.get(k, 2.0) * v for k, v in breakdown.items()]
        avg_footprint = sum(weights) / sum(breakdown.values())
    else:
        avg_footprint = 0.0
    footprint_norm = min(avg_footprint / 5.5, 1.0)

    # Signal 3: plate read confidence (best plate if multiple, else 0)
    real_plates = [p for p in plates if not p.get("mock", False)]
    plate_conf = max((p["score"] for p in real_plates), default=0.0)

    # Signal 4: EPI score from PS1 clustering (already 0-100, normalise to 0-1)
    hotspot = next((h for h in hotspots if h["cluster_id"] == cluster_id), None)
    epi_norm = (hotspot["congestion_impact_score"] / 100.0) if hotspot else 0.5

    # Weighted combination
    raw_score = (
        0.35 * yolo_conf
      + 0.20 * footprint_norm
      + 0.20 * plate_conf
      + 0.25 * epi_norm
    )
    final_score = round(raw_score * 100, 1)  # 0-100

    # Repeat-offender boost: if any plate is a known repeat offender,
    # add up to 10 points (capped at 100)
    if any(p.get("is_repeat_offender") for p in plates):
        final_score = min(final_score + 10.0, 100.0)

    return {
        "confidence_score": final_score,
        "signal_breakdown": {
            "yolo_detection_conf":  round(yolo_conf * 100, 1),
            "vehicle_footprint":    round(footprint_norm * 100, 1),
            "plate_read_conf":      round(plate_conf * 100, 1),
            "hotspot_epi":          round(epi_norm * 100, 1),
        },
        "plates_detected":      [p["plate"] for p in real_plates],
        "repeat_offender_flag": any(p.get("is_repeat_offender") for p in plates),
    }


print(f"{'Cluster':<10} {'Score':>6}  {'YOLO%':>6}  {'Foot%':>6}  {'Plate%':>7}  {'EPI%':>6}  {'Repeat':>7}  Plates")
print("-" * 80)

confidence_results = {}
for cid, det in detection_results.items():
    if not det:
        continue
    plates = plate_results.get(cid, [])
    result = compute_confidence_score(cid, det, plates)
    confidence_results[cid] = result

    sb = result["signal_breakdown"]
    print(
        f"[{cid}]{'':<6} {result['confidence_score']:>6}  "
        f"{sb['yolo_detection_conf']:>6}  "
        f"{sb['vehicle_footprint']:>6}  "
        f"{sb['plate_read_conf']:>7}  "
        f"{sb['hotspot_epi']:>6}  "
        f"{'YES' if result['repeat_offender_flag'] else 'no':>7}  "
        f"{result['plates_detected']}"
    )


In [ ]:
# Merge confidence scores + plate results into the visual summary and save
for cid_str, det in visual_summary.items():
    cid = int(cid_str)
    det["plates"]             = plate_results.get(cid, [])
    det["confidence"]         = confidence_results.get(cid, {})

out_path = PROCESSED / "hotspot_visual_summary.json"
with open(out_path, "w") as f:
    json.dump(visual_summary, f, indent=2, default=str)

print(f"Saved enriched visual summary: {out_path}")
print(f"Fields per entry: {list(next(iter(visual_summary.values())).keys())}")
